# 🔍 บทที่ 4 — SAM: ผู้วาดขอบเขตแปลง (Segment Anything Model)

> **สื่อการสอน Workshop:** การจำแนกพื้นที่ปลูกปาล์มน้ำมันด้วยภาพถ่ายดาวเทียมและ AI  
> ต่อจาก: [บทที่ 3 — Train Model](03_train_model.ipynb)  
> ⚡ **ต้องใช้ GPU** — ไปที่เมนู Runtime → Change runtime type → GPU

---

## บทที่ 4 นี้ เราจะทำอะไร?

ใน Workshop นี้ AI ที่เราสอน (บทที่ 3) จะทำหน้าที่ **ตัดสินว่า pixel ไหนเป็นปาล์ม**  
แต่เราต้องการ **เส้นขอบของแต่ละแปลง** ที่สวยงามและแม่นยำด้วย  

ที่มาของเส้นขอบเหล่านั้นคือ **SAM (Segment Anything Model)** — โมเดลจาก Meta AI ที่เชี่ยวชาญในการ 'วาดเส้นขอบ' วัตถุทุกอย่างในภาพ

<!-- 🖼️ โครงสร้างสำหรับสร้างรูปประกอบ -->
**โครงสร้างรูป:**
```text
บทบาทในระบบทั้งหมด:

  AI ของเรา (บทที่ 3)         SAM (บทที่ 4-5)
  ┌─────────────────┐         ┌─────────────────┐
  │ Pixel นี้ = ปาล์ม │  ✚   │  ขอบแปลงอยู่ตรงนี้  │
  │ Probability Map │         │  Polygon Outline │
  └─────────────────┘         └─────────────────┘
            ↓ รวมกัน (บทที่ 7)
  ┌─────────────────────────────────┐
  │  แผนที่แปลงปาล์มที่สมบูรณ์         │
  └─────────────────────────────────┘
```

**Prompt:**
`"A clean two-panel diagram. Left panel: a grayscale probability heatmap labeled 'AI ของเรา: รู้ว่าตรงไหนเป็นปาล์ม'. Right panel: a satellite image with clean polygon outlines labeled 'SAM: รู้ว่าขอบแปลงอยู่ตรงไหน'. Below both, a large combined map showing palm plantation polygons filled with mint-green #35C778. White background #FFFFFF, academic style, 16:9."`

ในบทนี้เราจะ:
1. ทำความเข้าใจว่า SAM ทำงานยังไง
2. เข้าใจว่าทำไม SAM ใช้กับ **ภาพ Google Satellite RGB** (ไม่ใช่ SAR)
3. ติดตั้งและโหลด SAM
4. ลองรัน SAM บนภาพตัวอย่าง
5. เข้าใจ Parameters สำคัญที่ควบคุมผลลัพธ์
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/A-Phrom-A/ssb-workshop/blob/main/04_sam_basics.ipynb)



---

## ทฤษฎี: SAM คืออะไร และทำงานอย่างไร?

**SAM (Segment Anything Model)** คือโมเดล AI จาก Meta AI ปี 2023  
ฝึกจากภาพ RGB กว่า **1,000 ล้านรูป** และ **1,100 ล้าน Mask**

---

### SAM1 ที่เราใช้: Automatic Mask Generation (AMG)

หัวใจของ SAM1 คือการ **กระจาย Grid ของ prompt points ทั่วทั้งภาพโดยอัตโนมัติ**  
ไม่ต้องคลิกหรือบอกตำแหน่งเอง — SAM วาง Grid แล้ว Segment ให้ทุกจุดเอง

<!-- 🖼️ โครงสร้างสำหรับสร้างรูปประกอบ -->
**โครงสร้างรูป:**
```text
SAM1 — Automatic Mask Generation (AMG)

  ภาพ Google Satellite (RGB)
          │
          ▼
  ┌─────────────────────────────────────┐
  │  กระจาย Grid of Prompt Points       │
  │                                     │
  │   . . . . . . . . . . . . . .       │  points_per_side = N
  │   . . . . . . . . . . . . . .       │  → N×N จุดทั่วภาพ
  │   . . . . . . . . . . . . . .       │
  │   . . . . . . . . . . . . . .       │
  └─────────────────────────────────────┘
          │
          ▼
  สร้าง Mask สำหรับทุกจุด (ขนานกัน)
          │
          ▼
  กรองคุณภาพ (pred_iou_thresh, stability_score_thresh)
          │
          ▼
  Segment Masks สุดท้าย (เส้นขอบวัตถุทุกชิ้นในภาพ)
```

**Prompt:**
`"A clean educational diagram showing SAM1 Automatic Mask Generation. LEFT: a Google Satellite image. CENTER: same image with a regular N×N dot grid overlaid across the entire image labeled 'Grid of Prompt Points (points_per_side=N)'. An arrow shows each dot automatically generates a mask. RIGHT: the same image filled with multiple colored segment regions labeled 'Generated Masks'. White background #FFFFFF, mint-green #35C778 accents, flat vector style, 16:9."`

### โครงสร้างภายใน (3 ส่วนของ SAM1)

```text
ภาพ RGB  ──►  [ Image Encoder (ViT-H) ]  ──►  Feature Map
                                                    │
Grid Point ──►  [ Prompt Encoder ]          ──►  Point Feature
                                                    │
                [ Mask Decoder ]  ◄── Feature Map + Point Feature
                       │
                Segment Mask (1 mask ต่อ 1 grid point)
```

| ส่วน | หน้าที่ | ความเร็ว |
|---|---|---|
| **Image Encoder (ViT-H)** | แปลงภาพ → Feature Map | ช้า (รัน **ครั้งเดียว** ต่อ 1 ภาพ) |
| **Prompt Encoder** | แปลง Grid Point → Feature | เร็วมาก |
| **Mask Decoder** | สร้าง Mask จาก 2 Feature | เร็วมาก (รันซ้ำ N×N ครั้ง) |

> 💡 **ความฉลาดของ SAM1:**  
> Image Encoder รันแค่ **ครั้งเดียว** แล้ว cache Feature Map ไว้  
> จากนั้น Mask Decoder รัน **N×N ครั้ง** (ทีละ grid point) โดยใช้ Feature เดิม  
> ทำให้แม้จะมี 64×64 = 4,096 จุด ก็ยังทำได้เร็วพอสมควร

---

### ความแตกต่าง SAM1 vs SAM2

| | SAM1 (ที่เราใช้) | SAM2 |
|---|---|---|
| **Mode หลัก** | Automatic Grid Mask Generation | Interactive + Video Tracking |
| **Prompt** | Grid points อัตโนมัติ | User click / กรอบ / Video frame |
| **ใช้งาน** | Segment ทุกอย่างในภาพ Batch | Real-time, Video, Interactive |
| **Library** | `samgeo` / `segment-geospatial` | `sam2` |


---

## SAM ใช้กับภาพอะไรใน Workshop นี้?

**ในงานของเรา** เราใช้ SAM กับ **ภาพถ่าย Google Satellite** ที่เป็น RGB ที่ความละเอียด Zoom Level 16  
ไม่ใช่ภาพ SAR — ภาพ SAR ใช้ในบทที่ 1-3 เพื่อ classify ว่า pixel ไหนเป็นปาล์มเท่านั้น

<!-- 🖼️ โครงสร้างสำหรับสร้างรูปประกอบ -->
**โครงสร้างรูป:**
```text
ใช้ข้อมูลต่างชนิดกัน เพื่อวัตถุประสงค์ต่างกัน

  ภาพ SAR (VV/VH)              ภาพ Google Satellite (RGB)
  ┌────────────┐               ┌────────────┐
  │░░▓▓▓▓▓░░░░│               │ สวนปาล์ม  │
  │░░▓▓▓▓▓░░░░│               │  (สีจริง) │
  └────────────┘               └────────────┘
  ↓ AI ของเรา (บทที่ 1-3)      ↓ SAM (บทที่ 4-5)
  'ตรงนี้เป็นปาล์ม'              'ขอบแปลงอยู่ที่นี่'
  Probability Map (0.0-1.0)     Polygon Segments
```

**Prompt:**
`"A comparison infographic. Left side: a dark grayscale SAR satellite image labeled 'SAR — ใช้หาปาล์ม (AI บทที่ 1-3)' with a heatmap overlay. Right side: a vivid color Google Maps satellite view of palm plantations labeled 'Google Satellite RGB — ใช้หาขอบแปลง (SAM บทที่ 4-5)' with clean polygon outlines. White background #FFFFFF, 16:9."`

> 💡 **ทำไม SAM ทำงานกับ Google Satellite ได้ดี?**  
> SAM ฝึกมาจากภาพ RGB ธรรมดา และ Google Satellite ก็เป็น RGB!  
> ขอบเขตแปลงปาล์มบนภาพดาวเทียมมีลักษณะ 'ขอบ' ที่ SAM มองเห็นได้ชัดเจน

---

## ขั้นตอนที่ 1 — ติดตั้งเครื่องมือ

In [ ]:
# =====================================================================
# ⚠️ สำหรับรันบน Local Machine (เครื่องตัวเอง)
# =====================================================================
import os
BASE_DIR = r'C:\Users\kittisak\Downloads\ssb tutor\drive_structure\ch04_sam_basics'


In [ ]:
# =====================================================================
# ☁️ สำหรับรันบน Google Colab (คอมเมนต์ไว้ก่อน)
# =====================================================================
# from google.colab import drive
# drive.mount('/content/drive')
# BASE_DIR = '/content/drive/MyDrive/ssb_workshop/ch04_sam_basics'


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1.1 — ติดตั้ง Library
# ═══════════════════════════════════════════════════════════

!pip install segment-geospatial geoai-py geopandas rasterio --quiet

print('✅ ติดตั้งเสร็จสิ้น')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1.2 — Import และ Mount Google Drive
# ═══════════════════════════════════════════════════════════

import gc
import os

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import rasterio
import torch
from samgeo import SamGeo


# ── Paths ──
RASTER_DIR = os.path.join(BASE_DIR, 'data')
OUTPUT_DIR = os.path.join(BASE_DIR, 'output')
MODEL_DIR  = os.path.join(BASE_DIR, 'models')
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# ภาพ Google Satellite สำหรับทดสอบ SAM
SAT_PATH = os.path.join(RASTER_DIR, 'for sam satelite test.tif')

# ตรวจสอบ GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ พร้อม! ใช้ Device: {device}')
print(f'   ภาพตัวอย่าง: {os.path.basename(SAT_PATH)}')

---

## ขั้นตอนที่ 2 — ดูภาพ Google Satellite ที่จะใช้

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2.1 — แสดงภาพ Google Satellite
# ═══════════════════════════════════════════════════════════

with rasterio.open(SAT_PATH) as src:
    print(f'ขนาดภาพ: {src.width} x {src.height} pixel')
    print(f'จำนวน Band: {src.count} (RGB = 3 bands)')
    print(f'ระบบพิกัด: {src.crs}')

    scale = min(1.0, 1500 / max(src.width, src.height))
    h = max(1, int(src.height * scale))
    w = max(1, int(src.width * scale))
    rgb = np.stack([src.read(i, out_shape=(h, w)) for i in range(1, 4)], axis=-1)
    if rgb.max() > 1:
        rgb = rgb / 255.0

fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(rgb)
ax.set_title('ภาพ Google Satellite (RGB) — SAM จะวาดขอบแปลงบนภาพนี้', fontsize=14, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

---

## ขั้นตอนที่ 3 — โหลด SAM Model

SAM มีให้เลือก 3 ขนาด:
| ขนาด | ชื่อ | ขนาดไฟล์ | ความเร็ว | ความแม่นยำ |
|---|---|---|---|---|
| ใหญ่ (แนะนำ) | `vit_h` | ~2.5 GB | ช้าสุด | ดีสุด |
| กลาง | `vit_l` | ~1.2 GB | ปานกลาง | ดี |
| เล็ก | `vit_b` | ~375 MB | เร็วสุด | พอใช้ |

> 💡 ถ้ายังไม่มีไฟล์ Model SAM จะ **Download อัตโนมัติ** จาก Meta AI เมื่อสั่ง `SamGeo()`

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3.1 — โหลด SAM Model (Auto-download ถ้ายังไม่มีไฟล์)
# ═══════════════════════════════════════════════════════════

sam = SamGeo(
    model_type='vit_h',          # ใช้ขนาดใหญ่สุด (แนะนำ)
    checkpoint_dir=MODEL_DIR,    # บันทึกไฟล์ model ไว้ใน Drive
    device=device,
)

print('✅ โหลด SAM เสร็จสิ้น!')

---

## ขั้นตอนที่ 4 — Parameters ของ SAM (Automatic Mask Generation)

เราใช้ **Automatic Mask Generation (AMG)** — SAM วาง 'ตาข่ายจุด' ทั่วภาพแล้วสร้าง Mask ให้ทุกจุดอัตโนมัติ

<!-- 🖼️ โครงสร้างสำหรับสร้างรูปประกอบ -->
**โครงสร้างรูป:**
```text
  ภาพ (1024x1024)     ตาข่าย points (points_per_side=8)

  ┌────────────┐        . . . . . . . .
  │            │        . . . . . . . .
  │            │  →     . . . . . . . .   → สร้าง Mask ทุกจุด
  │            │        . . . . . . . .
  └────────────┘        . . . . . . . .
```

**Prompt:**
`"A clean educational diagram showing Automatic Mask Generation. LEFT: a satellite image. CENTER: same image with regular dot grid overlaid labeled 'Prompt Points (points_per_side=8)'. RIGHT: same image with multiple colored polygon regions labeled 'Generated Masks'. Arrows connect steps. White background #FFFFFF, mint-green accents, 16:9."`

| Parameter | ความหมาย | ค่าสูง | ค่าต่ำ |
|---|---|---|---|
| `points_per_side` | จำนวนจุดต่อด้าน | Mask เล็กละเอียด | Mask ใหญ่รวม |
| `pred_iou_thresh` | คุณภาพขั้นต่ำ | เก็บเฉพาะ Mask ดีมาก | เก็บหลากหลาย |
| `stability_score_thresh` | ความคงเส้นคงวา | แน่นอน | หลากหลาย |
| `min_mask_region_area` | พื้นที่ขั้นต่ำ (pixel²) | ตัดเล็กทิ้ง | เก็บทุกขนาด |

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4.1 — ทดลองรัน SAM บนภาพตัวอย่าง
# ═══════════════════════════════════════════════════════════
# ⏱️ ใช้เวลาประมาณ 2-10 นาที ขึ้นกับขนาดภาพและ GPU

output_mask_tif  = os.path.join(OUTPUT_DIR, 'test_masks.tif')
output_mask_gpkg = os.path.join(OUTPUT_DIR, 'test_masks.gpkg')

sam_kwargs_test = {
    'points_per_side': 12,         # ทดสอบด้วยจำนวนจุดปานกลาง
    'points_per_batch': 8,
    'pred_iou_thresh': 0.75,
    'stability_score_thresh': 0.7,
    'min_mask_region_area': 500,
}

print('🔍 กำลังรัน SAM...')
sam.generate(
    SAT_PATH,
    output=output_mask_tif,
    foreground=True,
    unique=True,
    **sam_kwargs_test,
)
sam.tiff_to_vector(output_mask_tif, output_mask_gpkg)

gdf = gpd.read_file(output_mask_gpkg)
print(f'\n✅ SAM รันเสร็จ! พบ Segment ทั้งหมด: {len(gdf):,} ชิ้น')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4.2 — แสดงผลลัพธ์ SAM ซ้อนบนภาพ
# ═══════════════════════════════════════════════════════════

with rasterio.open(SAT_PATH) as src:
    bounds = src.bounds
    scale  = min(1.0, 1500 / max(src.width, src.height))
    h, w   = max(1, int(src.height * scale)), max(1, int(src.width * scale))
    rgb    = np.stack([src.read(i, out_shape=(h, w)) for i in range(1, 4)], axis=-1)
    if rgb.max() > 1:
        rgb = rgb / 255.0
    crs = src.crs

extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

axes[0].imshow(rgb, extent=extent, origin='upper')
axes[0].set_title('ก่อน: Google Satellite ต้นฉบับ', fontsize=13, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(rgb, extent=extent, origin='upper')
gdf.to_crs(crs).boundary.plot(ax=axes[1], color='#35C778', linewidth=0.6, alpha=0.8)
axes[1].set_title(f'หลัง: SAM วาดขอบได้ {len(gdf):,} Segment', fontsize=13, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()
print('\n💡 สังเกต: SAM วาดเส้นขอบทุกอย่าง ทั้งแปลงปาล์ม ถนน และพื้นที่อื่น')
print('   → บทถัดไปเราจะปรับ Parameters และกรองเอาเฉพาะส่วนที่ต้องการ')

---

## 🎯 สรุปบทที่ 4

| สิ่งที่เรียนรู้ | รายละเอียด |
|---|---|
| SAM คืออะไร | Foundation Model จาก Meta AI เชี่ยวชาญการวาดขอบเขต |
| ใช้กับอะไร | ภาพ Google Satellite (RGB) ไม่ใช่ SAR |
| บทบาทใน Workshop | วาดเส้นขอบแปลงให้สวยงาม → Overlay กับผลลัพธ์ AI |
| AMG คืออะไร | Automatic Mask Generation — SAM ทำงานเองทั่วภาพ |

> ➡️ **บทที่ 5:** รัน SAM สองแบบแล้วผสมกัน และตัดถนนออก